# Sesión 03 - Lab 1: COPY INTO

Este laboratorio usa `COPY INTO` para cargar ventas de tienda de forma incremental e idempotente, valida ese comportamiento reejecutando la misma carga sin archivos nuevos, y cierra mostrando en vivo el antipatrón de forzar una recarga con `COPY_OPTIONS ('force' = 'true')`. Antes de correrlo, sube `ventas_lote1.csv` al volume `/Volumes/dbassociate/default/vol_landing/sesion_03/ventas/` (arrastrando el archivo en el explorador de Catalog, o con `dbutils.fs.cp` si ya está en el workspace). `ventas_lote2.csv` se sube más adelante, en el Lab 1D.

## Verificación del entorno

In [0]:
# dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/sesion_03")
dbutils.fs.ls("abfss://metastore-data@saassociatedbkrs.dfs.core.windows.net/data/ventas/")


## Lab 1A: Crear la tabla destino con esquema explícito

Igual que con Auto Loader y la lectura batch de sesiones anteriores, se evita depender de la inferencia automática de tipos: la tabla se crea con columnas explícitas antes de cargar nada. Las columnas `ingestion_timestamp` y `source_file` siguen la misma convención de auditoría usada desde la Sesión 01.

In [0]:
# creamos la tabla, todo con sql, tmb podemos usar %sql
# definimos los tipos de datos
spark.sql("""
CREATE TABLE IF NOT EXISTS dbassociate.default.ventas_lab1 (
    venta_id INT,
    fecha_venta DATE,
    tienda_id INT,
    producto STRING,
    categoria STRING,
    cantidad INT,
    precio_unitario DOUBLE,
    monto_total DOUBLE,
    medio_pago STRING,
    ingestion_timestamp TIMESTAMP,
    source_file STRING
)
""")

print("Tabla creada (o ya existente): dbassociate.default.ventas_lab1")


## Lab 1B: Primera carga con COPY INTO

`COPY INTO` lee todo el contenido de la carpeta `ventas/` del Volume. La subconsulta agrega `ingestion_timestamp` y `source_file` (vía `_metadata.file_name`, nunca `input_file_name()`) al vuelo, igual que se hacía con `withColumn` en la Sesión 02, pero dentro de la sintaxis SQL de `COPY INTO`.

In [0]:
resultado_carga_inicial = spark.sql("""
COPY INTO dbassociate.default.ventas_lab1 -- se copiará a esta tabla, todos los campos deben tener los mismos tipos que lo definido.
FROM (
    SELECT
        CAST(venta_id AS INT) AS venta_id,
        CAST(fecha_venta AS DATE) AS fecha_venta,
        CAST(tienda_id AS INT) AS tienda_id,
        producto,
        categoria,
        CAST(cantidad AS INT) AS cantidad,
        CAST(precio_unitario AS DOUBLE) AS precio_unitario,
        CAST(monto_total AS DOUBLE) AS monto_total,
        medio_pago,
        current_timestamp() AS ingestion_timestamp,
        _metadata.file_name AS source_file
    -- FROM '/Volumes/dbassociate/default/vol_landing/sesion_03/ventas/' -- toma todos los archivos de la ruta para acumularlos
    FROM 'abfss://metastore-data@saassociatedbkrs.dfs.core.windows.net/data/ventas/' -- ruta en mi ADLS
)
FILEFORMAT = CSV -- 
FORMAT_OPTIONS ('header' = 'true')
""")

resultado_carga_inicial.show(truncate=False)
print("Filas en la tabla tras la primera carga:", spark.table("dbassociate.default.ventas_lab1").count())


## Lab 1C: Reejecutar el mismo COPY INTO (sin archivos nuevos)

Se vuelve a correr exactamente la misma sentencia, sin subir ningún archivo nuevo. `COPY INTO` reconoce que `ventas_lote1.csv` ya fue cargado y lo omite: `num_affected_rows` debería salir en 0.

In [0]:
resultado_reejecucion = spark.sql("""
COPY INTO dbassociate.default.ventas_lab1
FROM (
    SELECT
        CAST(venta_id AS INT) AS venta_id,
        CAST(fecha_venta AS DATE) AS fecha_venta,
        CAST(tienda_id AS INT) AS tienda_id,
        producto,
        categoria,
        CAST(cantidad AS INT) AS cantidad,
        CAST(precio_unitario AS DOUBLE) AS precio_unitario,
        CAST(monto_total AS DOUBLE) AS monto_total,
        medio_pago,
        current_timestamp() AS ingestion_timestamp,
        _metadata.file_name AS source_file
    -- FROM '/Volumes/dbassociate/default/vol_landing/sesion_03/ventas/'
    FROM 'abfss://metastore-data@saassociatedbkrs.dfs.core.windows.net/data/ventas/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
""")

# no se agrego nada nuevo, porque no se subio ningun archivo
resultado_reejecucion.show(truncate=False)
print("Filas en la tabla tras reejecutar sin archivos nuevos:", spark.table("dbassociate.default.ventas_lab1").count())


## Lab 1D: Carga incremental — sube `ventas_lote2.csv`

Sube `ventas_lote2.csv` a la misma carpeta `/Volumes/dbassociate/default/vol_landing/sesion_03/ventas/` y vuelve a correr la celda siguiente (la sentencia `COPY INTO` es idéntica a las anteriores). Esta vez `COPY INTO` detecta que solo `ventas_lote2.csv` es nuevo y carga únicamente esas filas.

In [0]:
resultado_incremental = spark.sql("""
COPY INTO dbassociate.default.ventas_lab1
FROM (
    SELECT
        CAST(venta_id AS INT) AS venta_id,
        CAST(fecha_venta AS DATE) AS fecha_venta,
        CAST(tienda_id AS INT) AS tienda_id,
        producto,
        categoria,
        CAST(cantidad AS INT) AS cantidad,
        CAST(precio_unitario AS DOUBLE) AS precio_unitario,
        CAST(monto_total AS DOUBLE) AS monto_total,
        medio_pago,
        current_timestamp() AS ingestion_timestamp,
        _metadata.file_name AS source_file
    -- FROM '/Volumes/dbassociate/default/vol_landing/sesion_03/ventas/'
    FROM 'abfss://metastore-data@saassociatedbkrs.dfs.core.windows.net/data/ventas/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
""")

# debe actualizar con los nuevos registros.
resultado_incremental.show(truncate=False)
print("Filas en la tabla tras cargar ventas_lote2.csv:", spark.table("dbassociate.default.ventas_lab1").count())


## Lab 1E: Antipatrón — forzar la recarga con `force = true`

`COPY_OPTIONS ('force' = 'true')` desactiva la idempotencia y recarga archivos que `COPY INTO` ya había marcado como procesados. Sirve para reprocesar datos corregidos en el origen, pero usado por costumbre (en vez de entender por qué una carga no avanzó) duplica filas. Se corre a propósito para ver el efecto, y se deshace recreando la tabla — un `TRUNCATE` no alcanza, porque no reinicia el tracking de archivos de `COPY INTO`.

In [0]:
resultado_forzado = spark.sql("""
COPY INTO dbassociate.default.ventas_lab1
FROM (
    SELECT
        CAST(venta_id AS INT) AS venta_id,
        CAST(fecha_venta AS DATE) AS fecha_venta,
        CAST(tienda_id AS INT) AS tienda_id,
        producto,
        categoria,
        CAST(cantidad AS INT) AS cantidad,
        CAST(precio_unitario AS DOUBLE) AS precio_unitario,
        CAST(monto_total AS DOUBLE) AS monto_total,
        medio_pago,
        current_timestamp() AS ingestion_timestamp,
        _metadata.file_name AS source_file
    -- FROM '/Volumes/dbassociate/default/vol_landing/sesion_03/ventas/'
    FROM 'abfss://metastore-data@saassociatedbkrs.dfs.core.windows.net/data/ventas/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('force' = 'true')
""")

# se duplica la data si forzamos sin eliminar la data.
resultado_forzado.show(truncate=False)
print("Filas tras forzar la recarga (duplicados esperados):", spark.table("dbassociate.default.ventas_lab1").count())

# Deshacer el antipatrón: recrear la tabla, no un TRUNCATE
spark.sql("DROP TABLE IF EXISTS dbassociate.default.ventas_lab1") # eliminamos la tabla

# la volvemos a crear
spark.sql("""
CREATE TABLE dbassociate.default.ventas_lab1 (
    venta_id INT,
    fecha_venta DATE,
    tienda_id INT,
    producto STRING,
    categoria STRING,
    cantidad INT,
    precio_unitario DOUBLE,
    monto_total DOUBLE,
    medio_pago STRING,
    ingestion_timestamp TIMESTAMP,
    source_file STRING
)
""")
spark.sql("""
COPY INTO dbassociate.default.ventas_lab1
FROM (
    SELECT
        CAST(venta_id AS INT) AS venta_id,
        CAST(fecha_venta AS DATE) AS fecha_venta,
        CAST(tienda_id AS INT) AS tienda_id,
        producto,
        categoria,
        CAST(cantidad AS INT) AS cantidad,
        CAST(precio_unitario AS DOUBLE) AS precio_unitario,
        CAST(monto_total AS DOUBLE) AS monto_total,
        medio_pago,
        current_timestamp() AS ingestion_timestamp,
        _metadata.file_name AS source_file
    FROM '/Volumes/dbassociate/default/vol_landing/sesion_03/ventas/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
""")
print("Tabla recreada sin duplicados, filas:", spark.table("dbassociate.default.ventas_lab1").count())


## Lab 1F: Observabilidad con DESCRIBE HISTORY

`DESCRIBE HISTORY` expone `operationMetrics` por cada operación registrada en el log de transacciones de la tabla. Para `COPY INTO`, incluye `numFiles` (archivos escritos) y `numOutputRows` (filas insertadas) — la misma fuente que confirmó la idempotencia de los Labs 1C y 1D, ahora leída directamente.

In [0]:
from pyspark.sql.functions import col

# existe un history por tabla, es decir, un history por ventas_lab1, ventas_lab2, ventas_lab3
# siempre se invoca por DESCRIBE HISTORY
historial = spark.sql("DESCRIBE HISTORY dbassociate.default.ventas_lab1")

(
    historial
    .filter("operation = 'COPY INTO'")
    .select(
        "version",
        "timestamp",
        col("operationMetrics")["numOutputRows"].alias("filas_insertadas"),
        col("operationMetrics")["numFiles"].alias("archivos_escritos"),
    )
    .orderBy("version")
).show(truncate=False)


## Limpieza

In [0]:
spark.sql("DROP TABLE IF EXISTS dbassociate.default.ventas_lab1")

print("Tabla temporal de este laboratorio eliminada.")
